# Martinez-Conde et al. (2019) on CRCNS pvc-12

Paper. Martinez-Conde S, McCamy MB, Troncoso XG, Otero-Millan J, Macknik SL (2019). Area V1
responses to illusory corner-folds in Vasarely's nested squares and the Alternating Brightness
Star illusions. PLoS ONE 14(3): e0210941. doi:10.1371/journal.pone.0210941. The claims file
(`claims/martinez-conde-2019.yaml`) was written from the version of record (PMC6438452, Europe PMC
full text: main text, Methods, legends, Table 1 and the legends of the supporting information).

Dataset. CRCNS pvc-12 (doi:10.6080/K05D8Q1T), read from the public S3 bucket `crcnsarchive`
at `pvc-12/live/` (`V1_Vasarely_data.zip`, streamed and opened in place). CRCNS data sets are not
versioned, so the version is recorded as `unversioned` (docs/archives.md, CRCNS); the bucket holds
only the `live/` copy, so there is no other version to choose from. The MD5 sums from the data
set's `checksums.md5` are printed below so that a later change can be detected; the zip read
here has MD5 `e1301fe463e72332e2303b2053ee0559` in that file (printed again from the bucket).

What is used. Each recording's ROI-aligned spike rate map (`Standard.smoothed_image`, spikes
per ms, x 1000 for spikes/s) and its stimulus, from which the luminance in the aligned frame is
sampled. The geometry (midpoints, blocks, baseline squares) is computed from the polygon
vertices stored in the files; every parameter is cited in the helper `vasarely.py`. Recordings
without an aligned map cannot be used (see the data check).

| Claim | Tested |
|---|---|
| neuron-counts | tested |
| spikes-per-condition | blocked (data_missing) |
| s4-baseline-no-difference | tested |
| fig4a-above-baseline | tested |
| fig4b-spike-rate-difference | tested |
| fig4b-pairwise-significance | tested |
| fig5-average-maps | tested |
| table1-edge, table1-p180, table1-s90, table1-s52, table1-s40 | tested |
| fig6-midpoint-representative | tested |
| fig7a-rms-contrast-fit | tested |
| fig7c-fold-edge-equivalence | tested |
| fig7c-fold-edge-significance | tested |
| fig7d-nonlinear | tested |
| fig7d-dog-similar | blocked (data_missing) |
| s3-latency-peak | blocked (data_missing) |

In [ ]:
import io
import itertools
import time
import resource

import fsspec
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats
from statsmodels.stats.multitest import multipletests

import vasarely as v

DATASET = "pvc-12"
VERSION = "unversioned"
t0 = time.time()
with fsspec.open(f"https://crcnsarchive.s3-us-west-2.amazonaws.com/{DATASET}/live/checksums.md5") as f:
    print(f.read().decode())
z = v.open_zip()
recordings = v.list_recordings(z)
recs = [v.load_recording(z, r) for r in recordings]
sheet = pd.read_excel(io.BytesIO(z.read(v.ROOT + "V1_Vasarely_data_SMC.xlsx")), header=None)
print(f"loaded {len(recs)} recordings in {time.time() - t0:.0f} s, peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

In [ ]:
pd.set_option("display.width", 200)
rec_df = pd.DataFrame([{k: r[k] for k in ("folder", "condition", "code", "cell", "monkey", "format", "has_standard")}
                       for r in recs])
print("Files in the zip:", len([n for n in z.namelist() if not n.endswith("/")]))
print("\nRecordings (SpikeRateMap files) by folder, format and whether they hold the aligned Standard map:")
print(rec_df.groupby(["folder", "format", "has_standard"]).size().to_string())
print("\nUnique cell codes per folder:", rec_df.groupby("folder")["cell"].nunique().to_dict())
print("Monkeys over unique cells:", rec_df.drop_duplicates("cell")["monkey"].value_counts().to_dict())

# the spreadsheet: cell codes and an X per condition
hdr = sheet.apply(lambda r: r.astype(str).str.fullmatch("Cell").any(), axis=1).idxmax()
cols = sheet.iloc[hdr].tolist()
sh = sheet.iloc[hdr + 1:].copy()
sh.columns = cols
sh = sh.dropna(subset=["Cell"])
sh["Cell"] = sh["Cell"].astype(str)
marks = sh[["Corners", "Star", "Edge-Compare"]].notna()
print(f"\nSpreadsheet: {len(sh)} cell codes, {int((marks.sum(1) > 0).sum())} with at least one condition; "
      f"per condition {marks.sum().to_dict()}")
print("Codes with no condition marked:", sh.loc[marks.sum(1) == 0, "Cell"].tolist())
fold_name = {"Corners": "Corners", "Star": "Star", "Edge-Compare": "Edge-compare"}
mismatch = []
for _, r in sh.iterrows():
    for col in ["Corners", "Star", "Edge-Compare"]:
        in_files = ((rec_df["folder"] == fold_name[col]) & (rec_df["cell"] == r["Cell"])).any()
        if in_files != pd.notna(r[col]):
            mismatch.append((r["Cell"], col, "file" if in_files else "sheet only"))
print("Spreadsheet against files, mismatches:", mismatch)

# duplicates: identical raw maps deposited twice
hashes = {}
for r in recs:
    if r["format"] == "v5":
        key = round(r["full_image_sum"], 9)
        hashes.setdefault(key, []).append(r["code"])
print("Recordings with identical total of the full rate map (possible duplicates):",
      [v_ for v_ in hashes.values() if len(v_) > 1])

# the two ABS Star recordings: their stimulus files are MATLAB 7.3 (HDF5) with another layout
for r in recs:
    if r["code"].startswith("ABS"):
        h = h5py.File(io.BytesIO(z.read(r["stim_path"])), "r")
        first = h[h["dat/poly_vert/x"][0, 0]]
        print(f"{r['code']}: map file {r['format']}, stimulus file MATLAB 7.3 with fields {sorted(h['dat'].keys())}; "
              f"{h['dat/poly_vert/x'].shape[1]} polygons of {first.shape[0]} vertices (closed), no aligned map")

# units: image = spike count / eye time (ms); the one recording that also holds eyetime shows it
r_et = [r for r in recs if "eyetime" in r][0]
et, im = r_et["eyetime"], r_et["full_image"]
cnt = im * et
print(f"\n{r_et['code']} holds `eyetime` (range {et.min():.0f} to {et.max():.0f}, integer ms samples at 1 kHz):"
      f" image x eyetime deviates from an integer by at most {np.abs(cnt[et > 0] - np.round(cnt[et > 0])).max():.1e};"
      f" nonzero image where eyetime is 0: {int(np.count_nonzero(im[et == 0]))}; total spikes {np.round(cnt).sum():.0f}")
print("So `image` is spikes per ms of eye time (x 1000 = spikes/s), not raw spike counts as the descriptor says.")

std = [r for r in recs if r["has_standard"]]
print("\nAligned-frame fit (Standard vertices to screen vertices): max residual "
      f"{max(r['frame_fit_error'] for r in std):.1e} DU over {len(std)} recordings; "
      "Star frames with several exact fits (symmetry):", sum(r["frame_candidates"] > 1 for r in std))
geo = {r["code"]: v.geometry(r) for r in std}
chk = []
for r in std:
    g, L = geo[r["code"]], r["lum"]
    for f, fd in g["folds"].items():
        for m in fd["mids"]:
            chk.append(dict(condition=r["condition"], what=f + " midpoint", lum=L[int(round(m[1])), int(round(m[0]))]))
    for x0, y0 in g["baseline"]:
        sub = L[int(round(y0)):int(round(y0)) + 16, int(round(x0)):int(round(x0)) + 16]
        chk.append(dict(condition=r["condition"], what="baseline square max", lum=np.nanmax(sub) if np.isfinite(sub).any() else np.nan))
chk = pd.DataFrame(chk)
print("\nStimulus luminance (grey level 0 to 100) sampled at the computed points:")
print(chk.groupby(["condition", "what"])["lum"].describe()[["count", "min", "50%", "max"]].to_string())
print("Baseline squares entirely off the display (left out):", int(chk["lum"].isna().sum()))
print("Corners: luminance on the chosen bright fold minus 10 DU to either side, and on the other tooth: ",
      np.round(np.array([geo[r['code']]["bright_contrast"] for r in std if r["condition"] == "corners"]).mean(0), 1))
print("ROI pixels off the display, fraction by recording (only > 0.01):",
      {r["code"]: round(float(r["offscreen"].mean()), 2) for r in std if r["offscreen"].mean() > 0.01})

rng = pd.DataFrame([dict(condition=r["condition"], shape=r["rate"].shape, min=np.nanmin(r["rate"]), max=np.nanmax(r["rate"]),
                         mean=np.nanmean(r["rate"]), covered=np.mean(r["rate_raw"][~r["offscreen"]] > 0)) for r in std])
print("\nAligned smoothed rate maps (spikes/s): shape, range, mean, fraction of pixels with a nonzero raw rate")
print(rng.groupby(["condition"]).agg(n=("min", "size"), shape=("shape", "first"), min=("min", "min"), max=("max", "max"),
                                     mean=("mean", "median"), covered_min=("covered", "min"), covered_med=("covered", "median")).to_string())
print("Corners Standard frame shapes:", pd.Series([str(r["rate"].shape) for r in std if r["condition"] == "corners"]).value_counts().to_dict())

# the baseline squares against the files' own BaseActivity, where a recording holds it
cmp_ = pd.DataFrame([dict(condition=r["condition"], mine=v.baseline(r["rate"], geo[r["code"]]), stored=r["base_stored"])
                     for r in std if np.isfinite(r["base_stored"])]).dropna()
for c, x in cmp_.groupby("condition"):
    print(f"baseline squares vs stored BaseActivity, {c} (n = {len(x)}): median {x.mine.median():.2f} vs {x.stored.median():.2f} spikes/s, "
          f"Spearman rho {stats.spearmanr(x.mine, x.stored)[0]:.2f}, median ratio {np.nanmedian(x.mine / x.stored.replace(0, np.nan)):.2f}")
# can the stored BaseActivity be reproduced by 16 x 16 DU squares on the aligned smoothed map? Search every
# position of one square, of a pair symmetric about the frame centre, and of four squares mirror-symmetric
# about it, for the placement that fits all recordings of a condition (same frame size) best
def box16(S):
    c = np.cumsum(np.cumsum(np.pad(S, ((1, 0), (1, 0))), 0), 1)
    return (c[16:, 16:] - c[:-16, 16:] - c[16:, :-16] + c[:-16, :-16]) / 256


for cond in ["square_edge", "star", "corners"]:
    rs = [r for r in std if r["condition"] == cond and np.isfinite(r["base_stored"])]
    shape = pd.Series([r["rate"].shape for r in rs]).mode()[0]
    rs = [r for r in rs if r["rate"].shape == shape]
    B = np.stack([box16(np.nan_to_num(r["rate_unmasked"])) for r in rs])
    b = np.array([r["base_stored"] for r in rs])[:, None, None]
    errs = {}
    for name, P in [("one square", B), ("point-symmetric pair", (B + B[:, ::-1, ::-1]) / 2),
                    ("four mirror-symmetric", (B + B[:, ::-1] + B[:, :, ::-1] + B[:, ::-1, ::-1]) / 4)]:
        errs[name] = round(float((np.abs(P - b).max(0) / b.max()).min()), 3)
    print(f"BaseActivity search, {cond} (n = {len(rs)}): smallest maximum error over recordings, as a fraction of the largest value:", errs)
print("Recordings whose polygon vertex order differs between map and stimulus file (identity order not an exact fit):",
      {c_: int(sum(r["identity_fit_error"] > 1e-6 for r in std if r["condition"] == c_)) for c_ in ["square_edge", "star", "corners"]})
print("smoothed / raw total over the aligned map (a normalised smoothing keeps it near 1): range",
      np.round(min(r["std_sum_ratio"] for r in std), 3), "to", np.round(max(r["std_sum_ratio"] for r in std), 3))

# registration check by a contrast: Corners has a bright and a dark S40 fold; the neurons were mostly
# ON-dominant (Methods), so a map registered to the stimulus should respond more at the bright midpoint
bd = []
for r in std:
    if r["condition"] != "corners":
        continue
    g = geo[r["code"]]
    bd.append((v.midpoint(r["rate"], g["folds"]["S40"]), v.midpoint(r["rate"], g["dark_S40"])))
bd = np.array(bd)
print(f"Registration contrast, Corners (n = {len(bd)}): bright S40 midpoint median {np.median(bd[:, 0]):.2f}, dark S40 midpoint "
      f"median {np.median(bd[:, 1]):.2f} spikes/s; bright higher in {int((bd[:, 0] > bd[:, 1]).sum())}; "
      f"Wilcoxon signed-rank p = {stats.wilcoxon(bd[:, 0], bd[:, 1]).pvalue:.2g}")

# registration check: within the Square and Edge ROI, the lit rectangle against the black half
# beside it, with the true map and with the map mirrored left-right (which swaps the two)
d_true, d_mirror = [], []
for r in std:
    if r["condition"] != "square_edge":
        continue
    L, M = r["lum"], r["rate"]
    lit = (L > 20) & (np.arange(L.shape[0])[:, None] < 160)
    dark = (L == 0) & (np.arange(L.shape[0])[:, None] < 160)
    d_true.append(np.nanmean(M[lit]) - np.nanmean(M[dark]))
    Mm = M[:, ::-1]
    d_mirror.append(np.nanmean(Mm[lit]) - np.nanmean(Mm[dark]))
print(f"\nRegistration check, Square and Edge (n = {len(d_true)}): rate on the lit rectangle minus the black area beside it, "
      f"median {np.median(d_true):.2f} spikes/s (positive in {np.sum(np.array(d_true) > 0)}); "
      f"with the map mirrored, median {np.median(d_mirror):.2f} (positive in {np.sum(np.array(d_mirror) > 0)}); "
      f"Wilcoxon signed-rank true vs mirrored p = {stats.wilcoxon(d_true, d_mirror).pvalue:.2g}")

The checks above, against the data descriptor and the paper:

- The descriptor says 53 neurons and two files per cell and condition; the zip holds 77
  recordings of 53 cell codes, matching the spreadsheet's marks. The spreadsheet lists 58 cell
  codes, five of them with no condition.
- `image` is a rate (spikes per ms of eye time), not a count; `smoothed_image_norm` is
  normalised to its maximum after subtracting `BaseActivity`, not "normalized by the amount of
  time". The files also hold an undocumented `Standard` struct (the ROI in stimulus coordinates,
  aligned to the preferred orientation), polygon vertices, and in some recordings a
  `BaseActivity`.
- 15 Star recordings (prefix ABD) have no `Standard` struct, and neither have the 2 with prefix
  ABS, whose stimulus files are MATLAB 7.3 files of another layout with polygons of 11 points,
  not the 24 vertices of the paper's 12-point star (one of their map files is MATLAB 7.3 too). The gaze-to-stimulus offset (the RF position) needed to build the aligned map is
  not stored for any recording, so these 17 cannot be placed on the stimulus and are left out.

Every per-neuron measure below is computed once here.

In [ ]:
LEVELS_LUM = np.arange(0, 101, 10)                     # Fig 7B, equal luminance intervals of 10%
RMS_LEVELS = 0.610 + 0.037 * np.arange(10)             # Results, Fig 7 section, paragraph 1
PAPER_FIT = (-0.34, 0.05, 0.95)                        # Fig 7A legend


def inverse_fit(c, a, b, cc):
    return -np.log((c - cc) / a) / b


def measure(r, mapname="rate", base_kind="squares"):
    g, M = geo[r["code"]], r[mapname]
    base = v.baseline(M, g) if base_kind == "squares" else r["base_stored"]
    out = []
    for f, fd in g["folds"].items():
        mid = v.midpoint(M, fd)
        halves = [v.blocks(M, a, b) - base for a, b in fd["halves"]]
        out.append(dict(code=r["code"], cell=r["cell"], monkey=r["monkey"], condition=r["condition"], fold=f,
                        base=base, mid=mid, srd=mid - base, halves=halves))
    return out


def per_neuron(mapname="rate", base_kind="squares", report=False):
    rows = []
    for r in std:
        if base_kind == "stored" and not np.isfinite(r["base_stored"]):
            continue
        rows += measure(r, mapname, base_kind)
    d = pd.DataFrame(rows)
    gone = d[~np.isfinite(d.base)].code.unique().tolist()
    if report:
        print(f"{mapname}, {base_kind}: recordings left out because every baseline square is off the display: {gone}")
    return d[np.isfinite(d.base)].reset_index(drop=True)


def edge_responses(levels, mapname="rate", base_kind="squares"):
    """Mean spike rate difference in 16 x 16 DU squares centred on the Edge at luminance levels (percent)."""
    out = {}
    for r in std:
        if r["condition"] != "square_edge":
            continue
        g, M = geo[r["code"]], r[mapname]
        base = v.baseline(M, g) if base_kind == "squares" else r["base_stored"]
        if np.isfinite(base):
            out[r["code"]] = [v.box_mean(M, *v.edge_point(g, l), v.EDGE_BOX) - base for l in levels]
    return pd.DataFrame(out, index=np.round(levels, 2)).T


FOLD_ORDER = ["P180", "S90", "S52", "S40"]
ANGLE = {"P180": 180, "S90": 90, "S52": 52, "S40": 40}
df = per_neuron(report=True)
df_raw = per_neuron("rate_raw", report=True)
df_st = per_neuron(base_kind="stored", report=True)
df_um = per_neuron("rate_unmasked", report=True)
print(df.groupby("fold").size().to_dict())


def sem(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return x.std(ddof=1) / np.sqrt(len(x))


def fold_table(d):
    return pd.DataFrame({f: dict(n=int((d.fold == f).sum()), mean=d.loc[d.fold == f, "srd"].mean(),
                                 sem=sem(d.loc[d.fold == f, "srd"])) for f in ["Edge"] + FOLD_ORDER}).T

## neuron-counts

> Of 118 V1 neurons recorded from two monkeys (102 from monkey J, 16 from monkey A), 60 were
> discarded for technical problems and 58 analysed; 7 were tested in all three conditions, 11 in
> two and 40 in one, giving 25 neurons for Square and Edge, 30 for Star and 28 for Corners.

In [ ]:
cells = rec_df.groupby("cell")["folder"].apply(set)
n_cond = cells.apply(len)
counts = pd.DataFrame({
    "expected": [58, 25, 30, 28, 7, 11, 40],
    "all files": [len(cells), (rec_df.folder == "Edge-compare").sum(), (rec_df.folder == "Star").sum(),
                  (rec_df.folder == "Corners").sum(), (n_cond == 3).sum(), (n_cond == 2).sum(), (n_cond == 1).sum()],
}, index=["neurons", "Square and Edge", "Star", "Corners", "three conditions", "two conditions", "one condition"])
cells_u = rec_df[rec_df.has_standard].groupby("cell")["folder"].apply(set)
nu = cells_u.apply(len)
counts["usable (aligned map)"] = [len(cells_u), ((rec_df.folder == "Edge-compare") & rec_df.has_standard).sum(),
                                  ((rec_df.folder == "Star") & rec_df.has_standard).sum(),
                                  ((rec_df.folder == "Corners") & rec_df.has_standard).sum(),
                                  (nu == 3).sum(), (nu == 2).sum(), (nu == 1).sum()]
counts["match"] = counts["expected"] == counts["all files"]
print(counts.to_string())
print("\nMonkeys over the dataset's cells:", rec_df.drop_duplicates("cell").monkey.value_counts().to_dict(),
      "(paper: 102 recorded from J and 16 from A, before 60 were discarded)")
print("Spreadsheet cell codes:", len(sh), "of which with a condition:", int((marks.sum(1) > 0).sum()))

**Result: not reproduced.** The data set holds 53 neurons in 77 recordings: 22 Square and Edge, 29
Star and 26 Corners, against the paper's 58, 25, 30 and 28. Of the overlap counts only the 7
neurons tested in all three conditions match (10 in two conditions against 11, 36 in one against
40). 49 of the 53 neurons are from monkey J and 4 from monkey A. The spreadsheet lists 58 cell
codes, but 5 have no condition marked and no files. Only 43 neurons, in 60 recordings (22 Square
and Edge, 12 Star, 26 Corners), have the aligned map that the other analyses need.
recordings: 22 Square and Edge, 29 Star and 26 Corners, against the paper's 58, 25, 30 and 28.
Of the overlap counts only the 7 neurons tested in all three conditions match (10 in two
against 11, 36 in one against 40). The spreadsheet lists 58 cell codes, but five carry no
condition and have no files.

## spikes-per-condition (blocked)

> Each experimental condition required at least 10,000 spikes, and the mean number of spikes per
> condition was 35,998 (STD 25,981).

The spike count map is not deposited: `image` is a rate, and the eye time needed to turn it
back into counts is present for one recording of 77 (printed in the data check: 15,644 spikes).
No spike times are deposited. The claim is blocked (`data_missing`).

## s4-baseline-no-difference

> Absolute baseline activity did not differ significantly across the three visual stimuli
> (Square and Edge, Star, Corners).

In [ ]:
def baseline_tests(d, label):
    b = d.drop_duplicates("code")[["code", "condition", "base"]]
    groups = {c: b.loc[b.condition == c, "base"].dropna().values for c in ["square_edge", "star", "corners"]}
    print(f"--- {label}")
    for c, x in groups.items():
        print(f"{c:12s} n={len(x):2d} median {np.median(x):6.2f}  quartiles {np.percentile(x, 25):6.2f} to "
              f"{np.percentile(x, 75):6.2f}  range {x.min():.2f} to {x.max():.2f} spikes/s")
    pairs = list(itertools.combinations(groups, 2))
    p = [stats.mannwhitneyu(groups[a], groups[b_], alternative="two-sided").pvalue for a, b_ in pairs]
    padj = multipletests(p, method="fdr_bh")[1]
    for (a, b_), pr, pa in zip(pairs, p, padj):
        print(f"(a) rank sum {a} vs {b_}: p = {pr:.3f}, BH-adjusted {pa:.3f}")
    kw = stats.kruskal(*groups.values())
    an = stats.f_oneway(*groups.values())
    print(f"(b) Kruskal-Wallis H = {kw.statistic:.2f}, p = {kw.pvalue:.3f}")
    print(f"(c) one-way ANOVA F = {an.statistic:.2f}, p = {an.pvalue:.3f}")
    return padj.min(), kw.pvalue, an.pvalue


res_s4 = baseline_tests(df, "baseline squares, smoothed map (main)")
baseline_tests(df_raw, "variant: unsmoothed map")
baseline_tests(df_st, "variant: the files' own BaseActivity (recordings that hold it)")
baseline_tests(df_um, "variant: off-display pixels not left out")
fig, ax = plt.subplots(figsize=(4, 3))
b = df.drop_duplicates("code")
ax.boxplot([b.loc[b.condition == c, "base"] for c in ["square_edge", "star", "corners"]], whis=(0, 100))
ax.set_xticks([1, 2, 3], ["Square and Edge", "Star", "Corners"])
ax.set_ylabel("absolute baseline (spikes/s)")
plt.show()
print("Every point is within the axes (whiskers at minimum and maximum).")

**Result: not reproduced.** All three readings find a difference in absolute baseline across the
stimuli: Square and Edge median 7.70 spikes/s (n = 22) against 2.31 for Star (n = 12) and 2.64
for Corners (n = 25); Kruskal-Wallis p = 0.005, ANOVA p = 0.001, and the BH-corrected rank sum
tests separate Square and Edge from Star (p = 0.036) and from Corners (p = 0.009), not Star from
Corners (p = 0.446). The unsmoothed map and the map without the off-display masking give the
same. With the baselines the files store themselves (`BaseActivity`, 9, 10 and 19 recordings) no
reading finds a difference (Kruskal-Wallis p = 0.114, smallest BH-adjusted p = 0.115); those
values are not reproduced by the paper's squares (median 7.03 against 5.64 spikes/s for Square
and Edge) and their computation is not documented.

## fig4a-above-baseline

> For every fold (P180, S90, S52, S40), most neurons' midpoint fold responses lie above their
> absolute baseline activity, and the mass center of each fold's points lies above the line of
> unit slope, with the mass center's distance from that line increasing as the fold angle
> decreases.

In [ ]:
def fig4a(d, label):
    print(f"--- {label}")
    rows = {}
    for f in FOLD_ORDER:
        x = d[d.fold == f]
        above = (x.mid > x.base).sum()
        pos = (x.mid > 0) & (x.base > 0)
        da = x.mid.mean() - x.base.mean()
        db = np.log10(x.mid[pos]).mean() - np.log10(x.base[pos]).mean()
        rows[f] = dict(n=len(x), above=int(above), frac=above / len(x), dist_a=da, dist_b_log10=db,
                       dropped_for_log=int((~pos).sum()))
    t = pd.DataFrame(rows).T
    print(t.to_string())
    ok_a = (t.frac > 0.5).all() and (t.dist_a > 0).all() and t.dist_a.is_monotonic_increasing
    ok_b = (t.frac > 0.5).all() and (t.dist_b_log10 > 0).all() and t.dist_b_log10.is_monotonic_increasing
    print("criterion met under reading (a):", ok_a, " under reading (b):", ok_b)
    return t


t4a = fig4a(df, "smoothed map, baseline squares (main)")
fig4a(df_raw, "variant: unsmoothed map")
fig, axs = plt.subplots(1, 4, figsize=(13, 3.2), sharex=True, sharey=True)
for ax, f in zip(axs, FOLD_ORDER):
    x = df[(df.fold == f) & (df["mid"] > 0) & (df["base"] > 0)]
    ax.loglog(x.base, x.mid, "b.")
    ax.plot([0.01, 300], [0.01, 300], "k:")
    ax.plot(x.base.mean(), x.mid.mean(), "r+", ms=12)
    ax.set_title(f)
    ax.set_xlabel("baseline (spikes/s)")
axs[0].set_ylabel("midpoint response (spikes/s)")
plt.show()
print("Points with a zero or negative value are not drawn on the log axes:",
      {f: int(((df.fold == f) & ~((df["mid"] > 0) & (df["base"] > 0))).sum()) for f in FOLD_ORDER})
print("Plotted range: baseline", round(df[df.base > 0].base.min(), 3), "to", round(df.base.max(), 1),
      "; midpoint", round(df[df.mid > 0].mid.min(), 3), "to", round(df.mid.max(), 1))

**Result: reproduced.** For every fold more than half of the neurons lie above the unit line
(P180 19 of 22, S90 20 of 22, S52 12 of 12, S40 25 of 25), every mass centre lies above it, and
its distance grows as the angle decreases under both readings: (a) 3.85, 8.09, 19.04 and 26.62
spikes/s; (b) 0.17, 0.30, 0.88 and 1.04 log10 units, with 2 S40 points that have a value of zero
or below left out of (b). The unsmoothed map gives the same order.

## fig4b-spike-rate-difference

> The mean spike rate difference at the fold midpoint decreases with increasing fold angle, from
> S40 (25.60 spikes/s) through S52 (18.10) and S90 (10.69) to P180 (4.87).

In [ ]:
EXP_MID = {"S40": (25.60, 4.18), "S52": (18.10, 2.39), "S90": (10.69, 1.94), "P180": (4.87, 1.15)}


def fig4b(d, label):
    t = fold_table(d).loc[FOLD_ORDER]
    t["expected"] = [EXP_MID[f][0] for f in FOLD_ORDER]
    t["tolerance"] = [EXP_MID[f][1] for f in FOLD_ORDER]
    t["within"] = (t["mean"] - t["expected"]).abs() <= t["tolerance"]
    order = t.loc["S40", "mean"] > t.loc["S52", "mean"] > t.loc["S90", "mean"] > t.loc["P180", "mean"]
    print(f"--- {label}")
    print(t.round(2).to_string())
    print("order S40 > S52 > S90 > P180:", order, "; all within tolerance:", bool(t.within.all()))
    return t, order


t4b, order4b = fig4b(df, "smoothed map, baseline squares (main)")
fig4b(df_raw, "variant: unsmoothed map")
fig4b(df_st, "variant: the files' own BaseActivity")
fig4b(df_um, "variant: off-display pixels not left out")
for m in ["J", "A"]:
    fig4b(df[df.monkey == m], f"monkey {m} only")

**Result: partially reproduced.** The order S40 > S52 > S90 > P180 holds (26.62, 19.04, 8.09 and
3.85 spikes/s), and S40, S52 and P180 are within the paper's SEM, but S90 (8.09) is below the
tolerance (10.69 - 1.94 = 8.75). The variants move single values across the bounds: the
unsmoothed map puts S90 in (9.49) and P180 out (2.91); the files' own baselines put S52 out
(20.90); monkey J alone (20, 20, 10 and 24 neurons) puts all four within. The Star value rests on
12 of the paper's 30 neurons.

## fig4b-pairwise-significance

> The midpoint spike rate difference for S40 is significantly larger than for S90 and for P180,
> and that for S52 is significantly larger than for P180.

In [ ]:
def pairwise(d, label):
    x = {f: d.loc[d.fold == f, "srd"].values for f in FOLD_ORDER}
    pairs = [("S40", "S90"), ("S40", "P180"), ("S52", "P180"), ("S40", "S52"), ("S52", "S90"), ("S90", "P180")]
    p = np.array([stats.mannwhitneyu(x[a], x[b_], alternative="two-sided").pvalue for a, b_ in pairs])
    pa6 = multipletests(p, method="fdr_bh")[1]
    pa3 = multipletests(p[:3], method="fdr_bh")[1]
    t = pd.DataFrame(dict(pair=[f"{a} vs {b_}" for a, b_ in pairs],
                          median_diff=[np.median(x[a]) - np.median(x[b_]) for a, b_ in pairs],
                          mean_diff=[x[a].mean() - x[b_].mean() for a, b_ in pairs],
                          p=p, p_bh_six=pa6, p_bh_three=list(pa3) + [np.nan] * 3))
    print(f"--- {label}")
    print(t.round(4).to_string(index=False))
    named = t.iloc[:3]
    ok_a = ((named.p_bh_six < 0.05) & (named.mean_diff > 0)).all()
    ok_b = ((named.p_bh_three < 0.05) & (named.mean_diff > 0)).all()
    print("three named pairs significant in the stated direction, (a) six-pair family:", ok_a, " (b) three-pair family:", ok_b)
    return t


tpw = pairwise(df, "smoothed map, baseline squares (main)")
pairwise(df_um, "variant: off-display pixels not left out")
pairwise(df_raw, "variant: unsmoothed map")
pairwise(df_st, "variant: the files' own BaseActivity")

**Result: reproduced.** S40 > S90 (BH-adjusted p = 0.0001), S40 > P180 and S52 > P180 (both
p < 0.0001) are significant under both families of comparisons. Of the other pairs, S52 > S90 is
significant (p = 0.0063 in the six-pair family), and S40 against S52 (p = 0.29) and S90 against
P180 (p = 0.083) are not. The three named pairs stay significant with the unsmoothed map, the
files' own baselines and without the off-display masking.

## fig5-average-maps

> The population-average spike rate difference maps show neural activity increasing as the
> corner angle decreases.

In [ ]:
avg = {}
for cond in ["square_edge", "star", "corners"]:
    rs = [r for r in std if r["condition"] == cond and np.isfinite(v.baseline(r["rate"], geo[r["code"]]))]
    # Maps are aligned on the midpoint of the condition's aligned fold (S90, S52, S40): the Corners
    # zigzag sits at a different place in each recording's ROI, and Corners maps come in two frame sizes.
    key = {"square_edge": "S90", "star": "S52", "corners": "S40"}[cond]
    ref = rs[0]
    shape = ref["rate"].shape
    ref_mid = np.mean(geo[ref["code"]]["folds"][key]["mids"], 0)
    maps, shifts = [], []
    for r in rs:
        M = r["rate"] - v.baseline(r["rate"], geo[r["code"]])
        dx, dy = np.round(ref_mid - np.mean(geo[r["code"]]["folds"][key]["mids"], 0)).astype(int)
        shifts.append((dx, dy))
        canvas = np.full(shape, np.nan)
        ys, xs = np.arange(M.shape[0]) + dy, np.arange(M.shape[1]) + dx
        ky, kx = (ys >= 0) & (ys < shape[0]), (xs >= 0) & (xs < shape[1])
        canvas[np.ix_(ys[ky], xs[kx])] = M[np.ix_(ky, kx)]
        maps.append(canvas)
    avg[cond] = dict(map=np.nanmean(np.stack(maps), 0), n=len(maps), ref=ref, left_out=0)
    print(cond, f"aligned on the {key} midpoint; shifts (dx, dy) range", np.min(shifts, 0), "to", np.max(shifts, 0))
vals = {}
for cond, f in [("square_edge", "P180"), ("square_edge", "S90"), ("star", "S52"), ("corners", "S40")]:
    A, ref = avg[cond]["map"], avg[cond]["ref"]
    fd = geo[ref["code"]]["folds"][f]
    mid = v.midpoint(A, fd)
    blk = np.nanmean([np.nanmean(v.blocks(A, a, b_)) for a, b_ in fd["halves"]])
    vals[f] = dict(n=avg[cond]["n"], left_out=avg[cond]["left_out"], midpoint=mid, ten_blocks=blk)
t5 = pd.DataFrame(vals).T
print(t5.round(2).to_string())
for col in ["midpoint", "ten_blocks"]:
    s = t5[col]
    print(col, "order S40 > S52 > S90 > P180:", bool(s["S40"] > s["S52"] > s["S90"] > s["P180"]))
fig, axs = plt.subplots(1, 3, figsize=(13, 5))
for ax, cond in zip(axs, ["square_edge", "star", "corners"]):
    A = avg[cond]["map"]
    imh = ax.imshow(A, cmap="hot", vmin=np.nanpercentile(A, 1), vmax=np.nanpercentile(A, 99))
    ax.contour(avg[cond]["ref"]["lum"], levels=[1, 50, 99], colors=["m"], linewidths=0.6)
    for fd in geo[avg[cond]["ref"]["code"]]["folds"].values():
        for m in fd["mids"]:
            ax.plot(*m, "c+")
    ax.set_title(f"{cond} (n = {avg[cond]['n']})")
    plt.colorbar(imh, ax=ax, fraction=0.04, label="spike rate difference (spikes/s)")
plt.show()
for cond in avg:
    A = avg[cond]["map"]
    print(f"{cond}: map range {np.nanmin(A):.1f} to {np.nanmax(A):.1f} spikes/s; colour scale clipped at the 1st and 99th percentiles")

**Result: reproduced.** In the population-average maps the midpoint values are S40 26.79, S52 19.04,
S90 8.09 and P180 3.85 spikes/s, and the means over the ten blocks are 23.15, 16.89, 7.22 and 3.22;
both readings are ordered by decreasing angle. As the claim notes, the midpoint reading repeats
the means of fig4b-spike-rate-difference and is not an independent test.

## Table 1: responses along the Edge and the folds

Claims table1-edge, table1-p180, table1-s90, table1-s52 and table1-s40. Readings: (a) per-neuron
summaries over its ten blocks, then the mean over neurons; (b) summaries of the population-average
profile. For S90 and S52, which have two halves: (i) the halves averaged block by block; (ii) each
half summarised, the two summaries averaged.

In [ ]:
EXP_T1 = {
    "Edge": [(7.98, 1.60), (33.13, 5.11), (20.44, 3.33), (20.77, 3.48), (22.01, 3.41)],
    "P180": [(0.18, 1.01), (10.28, 1.53), (4.55, 1.12), (4.03, 1.06), (4.87, 1.15)],
    "S90": [(4.36, 1.07), (14.61, 2.48), (9.22, 1.61), (9.31, 1.62), (10.69, 1.94)],
    "S52": [(6.26, 1.33), (25.39, 3.52), (15.66, 2.23), (15.93, 2.36), (18.10, 2.39)],
    "S40": [(9.33, 1.69), (35.11, 4.17), (22.81, 3.11), (22.83, 3.48), (25.60, 4.18)],
}
STATS = ["min", "max", "mean", "median"]


def summ(p):
    return np.array([np.nanmin(p), np.nanmax(p), np.nanmean(p), np.nanmedian(p)])


def table1(d, fold, verbose=True):
    x = d[d.fold == fold]
    mid = x.srd.mean()
    res = {}
    for halfmode in (["i", "ii"] if fold in ("S90", "S52") else ["i"]):
        # (a) per neuron
        if halfmode == "i":
            prof = np.stack([np.nanmean(np.stack(h), 0) for h in x.halves])
            per = np.stack([summ(p) for p in prof])
            pop = summ(np.nanmean(prof, 0))
        else:
            per = np.stack([np.mean([summ(p) for p in h], 0) for h in x.halves])
            nh = len(x.halves.iloc[0])
            pop = np.mean([summ(np.nanmean(np.stack([h[k] for h in x.halves]), 0)) for k in range(nh)], 0)
        res[f"(a{'' if halfmode == 'i' else ' ii'})"] = list(np.nanmean(per, 0)) + [mid]
        res[f"(b{'' if halfmode == 'i' else ' ii'})"] = list(pop) + [mid]
        res[f"sem (a{'' if halfmode == 'i' else ' ii'})"] = [sem(per[:, k]) for k in range(4)] + [sem(x.srd)]
    t = pd.DataFrame(res, index=STATS + ["midpoint"])
    t["expected"] = [e for e, _ in EXP_T1[fold]]
    t["tolerance"] = [tol for _, tol in EXP_T1[fold]]
    readings = [c for c in t.columns if c.startswith("(")]
    within = {c: int(((t[c] - t.expected).abs() <= t.tolerance).sum()) for c in readings}
    if verbose:
        print(f"--- {fold}: n = {len(x)}")
        print(t.round(2).to_string())
        print("values within tolerance (of 5), by reading:", within)
    best = max(within.values())
    return t, within, best


def outcome_t1(best):
    return "reproduced" if best == 5 else ("partially_reproduced" if best >= 3 else "not_reproduced")

In [ ]:
t1 = {}
for f in ["Edge"]:
    t1[f] = table1(df, f)
    print("outcome by the rule:", outcome_t1(t1[f][2]))
    print("variant, unsmoothed map:", table1(df_raw, f, verbose=False)[1])
    print("variant, the files' own BaseActivity:", table1(df_st, f, verbose=False)[1])
    print("variant, off-display pixels not left out:", table1(df_um, f, verbose=False)[1])

**Result: not reproduced.** Edge values are lower than Table 1's. Under reading (a): minimum 5.23,
maximum 29.16, mean 16.82, median 16.70 and midpoint 18.36 spikes/s, against 7.98, 33.13, 20.44,
20.77 and 22.01; only the maximum is within the tolerance. Under reading (b), only the median
(17.55) is. The variants reach at most 2 of 5.

In [ ]:
for f in ["P180"]:
    t1[f] = table1(df, f)
    print("outcome by the rule:", outcome_t1(t1[f][2]))
    print("variant, unsmoothed map:", table1(df_raw, f, verbose=False)[1])
    print("variant, the files' own BaseActivity:", table1(df_st, f, verbose=False)[1])
    print("variant, off-display pixels not left out:", table1(df_um, f, verbose=False)[1])

**Result: not reproduced.** P180 under reading (a): minimum -1.91, maximum 9.99, mean 3.22,
median 2.66 and midpoint 3.85 spikes/s, against 0.18, 10.28, 4.55, 4.03 and 4.87. The maximum and
the midpoint are within tolerance (2 of 5), and under reading (b) the median and the midpoint
are (2 of 5). With the files' own baselines (9 neurons) 4 of 5 are within, a variant only.

In [ ]:
for f in ["S90"]:
    t1[f] = table1(df, f)
    print("outcome by the rule:", outcome_t1(t1[f][2]))
    print("variant, unsmoothed map:", table1(df_raw, f, verbose=False)[1])
    print("variant, the files' own BaseActivity:", table1(df_st, f, verbose=False)[1])
    print("variant, off-display pixels not left out:", table1(df_um, f, verbose=False)[1])

**Result: not reproduced.** S90 under reading (a): minimum 2.14, maximum 13.21, mean 7.22, median
7.28 and midpoint 8.09 spikes/s, against 4.36, 14.61, 9.22, 9.31 and 10.69. Only the maximum is
within tolerance. No reading gives more than 1 of 5, and the variants give at most 3.

In [ ]:
for f in ["S52"]:
    t1[f] = table1(df, f)
    print("outcome by the rule:", outcome_t1(t1[f][2]))
    print("variant, unsmoothed map:", table1(df_raw, f, verbose=False)[1])
    print("variant, the files' own BaseActivity:", table1(df_st, f, verbose=False)[1])
    print("variant, off-display pixels not left out:", table1(df_um, f, verbose=False)[1])

**Result: reproduced.** S52 (12 neurons) under reading (a ii), with each half of the fold
summarised and the two summaries averaged: minimum 6.98, maximum 28.37, mean 16.84, median 16.31
and midpoint 19.04 spikes/s, against 6.26, 25.39, 15.66, 15.93 and 18.10. All five are within
tolerance. Reading (a) gives 4 of 5 (minimum 8.70 is out), and readings (b) and (b ii) give 3.

In [ ]:
for f in ["S40"]:
    t1[f] = table1(df, f)
    print("outcome by the rule:", outcome_t1(t1[f][2]))
    print("variant, unsmoothed map:", table1(df_raw, f, verbose=False)[1])
    print("variant, the files' own BaseActivity:", table1(df_st, f, verbose=False)[1])
    print("variant, off-display pixels not left out:", table1(df_um, f, verbose=False)[1])
fig, axs = plt.subplots(1, 5, figsize=(15, 3), sharey=True)
for ax, f in zip(axs, ["Edge", "P180", "S90", "S52", "S40"]):
    x = df[df.fold == f]
    prof = np.stack([np.nanmean(np.stack(h), 0) for h in x.halves])
    m, s = np.nanmean(prof, 0), np.nanstd(prof, 0, ddof=1) / np.sqrt(len(prof))
    ax.errorbar(np.arange(5, 100, 10), m, s, fmt="o-")
    ax.set_title(f"{f} (n = {len(x)})")
    ax.set_xlabel("block (% luminance)")
    print(f, "profile range", np.round(m.min(), 2), "to", np.round(m.max(), 2))
axs[0].set_ylabel("spike rate difference (spikes/s)")
plt.show()

**Result: reproduced.** S40 (25 neurons) under reading (a): minimum 10.33, maximum 34.19, mean
23.21, median 23.35 and midpoint 26.62 spikes/s, against 9.33, 35.11, 22.81, 22.83 and 25.60. All
five are within tolerance. Reading (b) gives 3 of 5.

## fig6-midpoint-representative

> For each fold and the Edge, the spike rate difference at the midpoint does not differ
> significantly from the mean or the median spike rate difference along the fold or Edge, so
> the midpoint response represents the whole fold.

In [ ]:
def fig6(d, label):
    rows = []
    for f in ["Edge", "P180", "S90", "S52", "S40"]:
        x = d[d.fold == f]
        for halfmode in (["i", "ii"] if f in ("S90", "S52") else ["i"]):
            if halfmode == "i":
                prof = [np.nanmean(np.stack(h), 0) for h in x.halves]
                mean_ = np.array([np.nanmean(p) for p in prof])
                med_ = np.array([np.nanmedian(p) for p in prof])
            else:
                mean_ = np.array([np.mean([np.nanmean(p) for p in h]) for h in x.halves])
                med_ = np.array([np.mean([np.nanmedian(p) for p in h]) for h in x.halves])
            for what, y in [("mean", mean_), ("median", med_)]:
                rows.append(dict(fold=f, halves=halfmode, against=what, n=len(x),
                                 median_diff=np.median(x.srd.values - y),
                                 p_ranksum=stats.mannwhitneyu(x.srd.values, y, alternative="two-sided").pvalue,
                                 p_signrank=stats.wilcoxon(x.srd.values, y).pvalue))
    t = pd.DataFrame(rows)
    print(f"--- {label}")
    print(t.round(4).to_string(index=False))
    main = t[t.halves == "i"]
    print("rank sum tests significant at 0.05 (halves reading i):", int((main.p_ranksum < 0.05).sum()), "of", len(main),
          "; with reading ii for S90 and S52:", int((t[(t.halves == 'ii') | ~t.fold.isin(['S90', 'S52'])].p_ranksum < 0.05).sum()),
          "; signed-rank variant significant:", int((main.p_signrank < 0.05).sum()))
    return t


t6 = fig6(df, "smoothed map, baseline squares (main)")
fig6(df_raw, "variant: unsmoothed map")
fig6(df_um, "variant: off-display pixels not left out")

**Result: reproduced.** None of the ten rank sum tests of the midpoint against the per-neuron mean
or median along the fold is significant (smallest p = 0.415, S40 median), whichever way the S90
and S52 halves are combined. The paired signed-rank variant finds 3 of the 10 significant (S90
median p = 0.025, S40 mean p = 0.015, S40 median p = 0.0067), with the midpoint above the fold
average.

## fig7a-rms-contrast-fit

> Local RMS contrast along the Edge rises from 0.61 to 0.943 with luminance and is fit by an
> exponential model, f(l) = -0.34 exp(-0.05 l) + 0.95, with R2 = 0.9983.

In [ ]:
LEV = np.arange(0, 100.01, 1.5)
curves = {"a": [], "b": []}
for r in std:
    if r["condition"] != "square_edge":
        continue
    g, L = geo[r["code"]], r["lum"]
    for reading in "ab":
        cc = []
        for l in LEV:
            cx, cy = v.edge_point(g, l)
            c0, r0 = int(np.floor(cx - 8 + 0.5)), int(np.floor(cy - 8 + 0.5))
            patch = L[max(r0, 0):r0 + 16, max(c0, 0):c0 + 16]
            cc.append(v.rms_contrast(patch, reading))
        curves[reading].append(cc)


def model(l, a, b, c):
    return a * np.exp(-b * l) + c


def fit_lar(l, y):
    p0, _ = optimize.curve_fit(model, l, y, p0=(-0.34, 0.05, 0.95), method="lm", maxfev=20000)
    res = optimize.minimize(lambda p: np.abs(model(l, *p) - y).sum(), p0, method="Nelder-Mead",
                            options=dict(xatol=1e-10, fatol=1e-12, maxiter=20000))
    p = res.x
    r2 = 1 - ((y - model(l, *p)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return p, r2


fit7a = {}
for reading in "ab":
    C = np.array(curves[reading])
    y = np.nanmedian(C, 0)
    p, r2 = fit_lar(LEV, y)
    fit7a[reading] = dict(lowest=y.min(), highest=y.max(), at_0=y[0], at_100=y[-1], a=p[0], b=p[1], c=p[2], R2=r2,
                          spread_over_recordings=np.nanmax(np.nanmax(C, 0) - np.nanmin(C, 0)))
t7a = pd.DataFrame(fit7a).T
t7a.loc["expected"] = [0.61, 0.943, np.nan, np.nan, -0.34, 0.05, 0.95, 0.9983, np.nan]
print(t7a.round(4).to_string())
tol = dict(lowest=0.061, highest=0.094, a=0.034, b=0.005, c=0.095, R2=0.0017)
for reading in "ab":
    within = {k: bool(abs(fit7a[reading][k] - t7a.loc["expected", k]) <= tl) for k, tl in tol.items()}
    print(f"reading ({reading}) within tolerance:", within)
fig, ax = plt.subplots(figsize=(5, 3))
for reading in "ab":
    ax.plot(LEV, np.nanmedian(np.array(curves[reading]), 0), ".", label=f"reading ({reading})")
ax.plot(LEV, model(LEV, *PAPER_FIT), "k-", label="paper's fit")
ax.set_xlabel("Edge luminance (%)")
ax.set_ylabel("local RMS contrast")
ax.legend()
plt.show()
print("plotted range, reading (a):", np.round(np.nanmin(curves['a']), 3), "to", np.round(np.nanmax(curves['a']), 3))

**Result: not reproduced.** Neither definition of RMS contrast gives the paper's curve along the
Edge. With reading (a), the standard deviation over the mean, the values run from 0.96 to 2.11
and do not rise with luminance (2.11 at 0% and 1.51 at 100%); the exponential fit has R2 0.83.
With reading (b), the standard deviation of normalised luminance, they run from 0.014 to 0.476;
the fit has R2 0.998, but its parameters (a = -15.9, b = 0.0003, c = 15.9) are far from the
paper's. The 16 DU square centred on the Edge is half black background, which neither reading
can turn into 0.61 to 0.943.

## fig7c-fold-edge-equivalence

> Neural responses to P180 are roughly equivalent to those to a 50% contrast Edge, while responses
> to S40 are close to those to a 90% contrast Edge, so sharper corners produce responses
> equivalent to higher contrasts along a straight edge.

In [ ]:
lum_for_rms = inverse_fit(RMS_LEVELS, *PAPER_FIT)
print("Luminance (%) of the ten RMS levels by the inverse of the paper's fit:", np.round(lum_for_rms, 1))
E_lum = edge_responses(LEVELS_LUM)
E_rms = edge_responses(lum_for_rms)
fold_means = fold_table(df).loc[FOLD_ORDER, "mean"]
print(f"Edge responses at 10% luminance steps (mean, SEM, n = {len(E_lum)}):")
print(pd.DataFrame(dict(mean=E_lum.mean(), sem=E_lum.apply(sem))).T.round(2).to_string())
print("Edge responses at the ten RMS levels:")
print(pd.DataFrame(dict(rms=RMS_LEVELS, mean=E_rms.mean().values, sem=E_rms.apply(sem).values)).T.round(3).to_string())
print("Fold midpoint means:", fold_means.round(2).to_dict())


def nearest(level_values, means, target):
    i = int(np.argmin(np.abs(means - target)))
    return level_values[i]


res7c = {}
na_p = nearest(LEVELS_LUM, E_lum.mean().values, fold_means["P180"])
na_s = nearest(LEVELS_LUM, E_lum.mean().values, fold_means["S40"])
res7c["a"] = (na_p, na_s, abs(na_p - 50) <= 10, abs(na_s - 90) <= 10)
nb_p = nearest(RMS_LEVELS * 100, E_rms.mean().values, fold_means["P180"])
nb_s = nearest(RMS_LEVELS * 100, E_rms.mean().values, fold_means["S40"])
res7c["b"] = (nb_p, nb_s, np.isclose(nb_p, 61.0), abs(nb_s - 90) <= 3.7)
for k, (p_, s_, okp, oks) in res7c.items():
    print(f"reading ({k}): Edge level nearest P180 = {p_:.1f}, nearest S40 = {s_:.1f}; P180 condition {okp}, S40 condition {oks}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.errorbar(RMS_LEVELS * 100, E_rms.mean(), E_rms.apply(sem), fmt="go", label="Edge at RMS levels")
for f in FOLD_ORDER:
    ax.axhline(fold_means[f], ls="--", lw=0.8)
    ax.text(61, fold_means[f], f)
ax.set_xlabel("RMS contrast x 100")
ax.set_ylabel("spike rate difference (spikes/s)")
plt.show()
print("plotted range: Edge", np.round(E_rms.mean().min(), 2), "to", np.round(E_rms.mean().max(), 2))
# variant: refit of the RMS curve from the data
for reading in "ab":
    p_ref = (fit7a[reading]["a"], fit7a[reading]["b"], fit7a[reading]["c"])
    with np.errstate(invalid="ignore", divide="ignore"):
        lum_ref = inverse_fit(RMS_LEVELS, *p_ref)
    print(f"variant, inverse of the refit under reading ({reading}) at the ten RMS levels:", np.round(lum_ref, 1),
          "(NaN where the level is outside the refit curve's range)")

**Result: not reproduced.** The Edge responses (8.34 to 20.09 spikes/s over the luminance levels)
lie above the P180 mean (3.85) and below the S40 mean (26.62) at every level. Under reading (a)
the Edge level nearest S40 is 90% (20.09) but the level nearest P180 is 0% (8.34), not within 10%
of 50%. Under reading (b) P180 matches the lowest RMS level (61.0), as the rule allows, but the
level nearest S40 is 94.3 (19.96), 4.3 from 90 against a step of 3.7. Neither reading meets both
conditions.

## fig7c-fold-edge-significance

> Responses to P180 are significantly lower than responses to a 94% contrast Edge, and responses
> to S40 are significantly higher than responses to an Edge of less than 68% contrast.

In [ ]:
rows = []
for f in FOLD_ORDER:
    x = df.loc[df.fold == f, "srd"].values
    for k, lvl in enumerate(RMS_LEVELS):
        y = E_rms.iloc[:, k].values
        t_ = stats.ttest_ind(x, y)
        rows.append(dict(fold=f, rms=round(lvl, 3), diff=x.mean() - y.mean(), t=t_.statistic, df=len(x) + len(y) - 2, p=t_.pvalue))
T = pd.DataFrame(rows)
T["bh_all40"] = multipletests(T.p, method="fdr_bh")[1]
T["bh_per_fold"] = T.groupby("fold").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
T["bonf_all40"] = np.minimum(T.p * len(T), 1)
T["bonf_per_fold"] = np.minimum(T.p * 10, 1)
print(T.round(4).to_string(index=False))
for col in ["bh_all40", "bh_per_fold", "bonf_all40", "bonf_per_fold"]:
    p180 = T[(T.fold == "P180") & np.isclose(T.rms, 0.943)].iloc[0]
    s40 = T[(T.fold == "S40") & T.rms.isin([0.61, 0.647])]
    ok = (p180[col] < 0.05 and p180["diff"] < 0) and ((s40[col] < 0.05) & (s40["diff"] > 0)).all()
    sig = T[T[col] < 0.05][["fold", "rms"]].values.tolist()
    print(f"{col}: P180 < Edge@0.943 p = {p180[col]:.4f}; S40 > Edge@0.610/0.647 p = {s40[col].round(4).tolist()}; criterion met: {ok}")
    print("   significant pairs:", sig)

**Result: reproduced.** P180 is significantly lower than the Edge at RMS 0.943 (difference -16.1
spikes/s; adjusted p = 0.0076 with BH over 40 tests, 0.020 with Bonferroni over 40), and S40 is
significantly higher than the Edge at 0.610 and 0.647 (differences 18.3 and 17.6; BH over 40
p = 0.0076 for both, Bonferroni over 40 p = 0.018 and 0.026). All four corrections meet the
criterion. With Bonferroni over 40, S40 is significant at 0.610 to 0.684 and not above; with BH,
up to 0.832.

## fig7d-nonlinear

> The relationship between corner angle and V1 spike rate difference is parametric but apparently
> not linear; the paper says it cannot be certain of this, given the few angles tested.

In [ ]:
ft = fold_table(df).loc[FOLD_ORDER]
ang = np.array([ANGLE[f] for f in FOLD_ORDER], float)
m_, s_ = ft["mean"].values.astype(float), ft["sem"].values.astype(float)
coef = np.polyfit(ang, m_, 1)
resid = m_ - np.polyval(coef, ang)
print(pd.DataFrame(dict(angle=ang, mean=m_, sem=s_, line=np.polyval(coef, ang), residual=resid,
                        off_by_more_than_sem=np.abs(resid) > s_), index=FOLD_ORDER).round(2).to_string())
print(f"line: slope {coef[0]:.3f} spikes/s per degree, intercept {coef[1]:.2f}; "
      f"R2 {1 - (resid ** 2).sum() / ((m_ - m_.mean()) ** 2).sum():.3f}")
monotonic = bool(m_[3] > m_[2] > m_[1] > m_[0])
print("monotonic (S40 > S52 > S90 > P180):", monotonic, "; non-linear by the SEM rule:", bool((np.abs(resid) > s_).any()))
o = np.argsort(ang)
a_s, m_s, s_s = ang[o], m_[o], s_[o]
slopes = np.diff(m_s) / np.diff(a_s)
slope_se = np.sqrt(s_s[1:] ** 2 + s_s[:-1] ** 2) / np.diff(a_s)
print("variant, slopes 40-52, 52-90, 90-180:", np.round(slopes, 3), "+/-", np.round(slope_se, 3))
for k in range(2):
    d_ = abs(slopes[k + 1] - slopes[k])
    print(f"  slopes {k} and {k + 1} differ by {d_:.3f}, propagated SEM {np.hypot(slope_se[k], slope_se[k + 1]):.3f}:",
          bool(d_ > np.hypot(slope_se[k], slope_se[k + 1])))

**Result: reproduced.** The four means fall monotonically with angle (26.62, 19.04, 8.09 and 3.85
spikes/s). The least-squares line (slope -0.144 spikes/s per degree, R2 0.778) misses three of
them by more than their SEM: P180 by +2.36 (SEM 0.83), S90 by -6.38 (SEM 1.92) and S40 by +4.94
(SEM 4.31). In the variant, the slopes for 40 to 52, 52 to 90 and 90 to 180 degrees are -0.632,
-0.288 and -0.047 spikes/s per degree; the last two differ by 0.241 against a propagated SEM of
0.099, the first two by 0.344 against 0.453.

## fig7d-dog-similar and s3-latency-peak (blocked)

fig7d-dog-similar needs the DOG model output of the paper's reference [3]; it is shown only as a
line in Fig 7D, and no numeric output is deposited in the data set or named as public by the
paper. s3-latency-peak needs spike times and eye positions to re-register spikes at other
latencies; the data set holds only maps made at the paper's latency.

## Summary

| Claim | Outcome |
|---|---|
| neuron-counts | not reproduced |
| spikes-per-condition | blocked: data missing (no spike counts or times) |
| s4-baseline-no-difference | not reproduced |
| fig4a-above-baseline | reproduced |
| fig4b-spike-rate-difference | partially reproduced (order holds; S90 below tolerance) |
| fig4b-pairwise-significance | reproduced |
| fig5-average-maps | reproduced |
| table1-edge | not reproduced |
| table1-p180 | not reproduced |
| table1-s90 | not reproduced |
| table1-s52 | reproduced |
| table1-s40 | reproduced |
| fig6-midpoint-representative | reproduced |
| fig7a-rms-contrast-fit | not reproduced |
| fig7c-fold-edge-equivalence | not reproduced |
| fig7c-fold-edge-significance | reproduced |
| fig7d-nonlinear | reproduced |
| fig7d-dog-similar | blocked: data missing (DOG model output) |
| s3-latency-peak | blocked: data missing (no spike times or eye positions) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. It starts from the deposited ROI-aligned rate maps,
not from spike times and eye positions, so the eye-signal correction, the latency and the
registration are the authors' own and cannot be checked. The smoothed map is the deposited one.
Star rests on 12 of the paper's 30 neurons, Square and Edge on 22 of 25 and Corners on 25 of 28
(one Corners recording has all its baseline squares off the display). Pixels of the ROI that fall
off the display are left out; keeping them changes no outcome. The Corners baseline squares are
placed where Fig 2C draws them, since the text gives no position.

What was found about the data set. The files hold 77 recordings of 53 neurons, against the
paper's 83 recordings of 58. `image` is a rate (spikes per ms of eye time), not a count as the
descriptor says, and `smoothed_image_norm` is normalised to its maximum after subtracting
`BaseActivity`. The aligned `Standard` maps, the polygon vertices, `BaseActivity` and `eyetime`
are undocumented. 17 of the 29 Star recordings cannot be placed on the stimulus: 15 lack the
aligned map, and 2 lack it and have MATLAB 7.3 stimulus files of another layout (polygons of 11 points, not 24). No spike times,
eye positions or eye time maps are deposited (`eyetime` is in one recording only).

In [ ]:
print(f"total run time {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")